# EnergyPredict — Training Demand Models

This notebook downloads data generated in **EnergyPredict_Dataset_Creation_Notebook.ipynb** notebook. That notebook downloaded data and built features, saving everything to `energy_data_final.csv`. Here, we use that file to actually train models that predict demand 24 hours in advance, and see which one is best.

**We'll try 4 approaches, from simplest to most complex:**
1. A "baseline" that doesn't really predict anything clever — just a sanity check
2. Ridge Regression — a simple straight-line-style model
3. Random Forest — lots of decision trees voting together
4. Gradient Boosting — decision trees that learn from each other's mistakes
5. LSTM — a neural network built to understand sequences over time

**The rule for all of them:** whoever has the lowest error wins. We're not assuming the fanciest model is automatically the best.

## Load the data we prepared earlier

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("energy_data_final.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

print("Rows:", len(df))
df.head()


## Split into "train" and "test" data

This is one of the most important ideas in machine learning: we teach the model using older data (**train**), then check how well it does on newer data it has never seen (**test**).

For time-based data like this, we must split by **date**, not randomly. If we let the model peek at random rows, some of those rows would come from *after* the test period, which would be like letting a student see the exam questions before studying. The results would look great but be fake.

In [ ]:
# Use the last 60 days as the test set, everything before that as training
cutoff_date = df["timestamp"].max() - pd.Timedelta(days=60)

train = df[df["timestamp"] < cutoff_date].reset_index(drop=True)
test = df[df["timestamp"] >= cutoff_date].reset_index(drop=True)

print("Training rows:", len(train), "  (", train['timestamp'].min().date(), "to", train['timestamp'].max().date(), ")")
print("Testing rows: ", len(test), "  (", test['timestamp'].min().date(), "to", test['timestamp'].max().date(), ")")


## Pick our input columns (features) and what we're predicting (target)

We use every column we built earlier *except* the ones that would be "cheating" — like the two target columns themselves, or the raw timestamp text.

In [ ]:
columns_to_exclude = ["timestamp", "target_t_plus_24h", "target_t_plus_48h"]
feature_columns = [col for col in df.columns if col not in columns_to_exclude]

X_train = train[feature_columns]
y_train = train["target_t_plus_24h"]

X_test = test[feature_columns]
y_test = test["target_t_plus_24h"]

print("Number of features:", len(feature_columns))


## A simple way to score each model

We'll use two easy-to-understand error measurements:
- **MAE (Mean Absolute Error)** — on average, how many MW off was the prediction? Smaller is better.
- **MAPE (Mean Absolute Percentage Error)** — the same idea, but as a percentage of the real value. Easier to compare across different problems.

In [ ]:
def score_model(actual_values, predicted_values):
    errors = actual_values - predicted_values
    mae = np.mean(np.abs(errors))
    mape = np.mean(np.abs(errors / actual_values)) * 100
    return mae, mape

# We'll collect every model's score here so we can compare them at the end
results = []


## The baseline — "tomorrow will look like today"

Before trying anything fancy, always check the simplest possible guess. Here, our guess for demand 24 hours from now is just... **whatever demand is right now.** Since 24 hours later is the *same hour of day* and any real model needs to beat it to prove it's worth using.

In [ ]:
baseline_prediction = test["demand_MW"].values

baseline_mae, baseline_mape = score_model(y_test.values, baseline_prediction)
results.append(("Baseline (persistence)", baseline_mae, baseline_mape))

print("Baseline MAE:", round(baseline_mae, 1), "MW")
print("Baseline MAPE:", round(baseline_mape, 2), "%")


## Ridge Regression

This model tries to draw the best possible straight-line-style relationship between all our features and demand. It's simple and fast, and a good sanity check before trying anything complicated.

In [ ]:
from sklearn.linear_model import Ridge

ridge_model = Ridge(alpha=10.0)
ridge_model.fit(X_train, y_train)
ridge_prediction = ridge_model.predict(X_test)

ridge_mae, ridge_mape = score_model(y_test.values, ridge_prediction)
results.append(("Ridge Regression", ridge_mae, ridge_mape))

print("Ridge MAE:", round(ridge_mae, 1), "MW")
print("Ridge MAPE:", round(ridge_mape, 2), "%")


## Random Forest

Imagine asking 300 different decision trees to each make a guess, then averaging all their guesses together. Each tree only sees a random slice of the data, so no single tree can dominate. This approach usually beats a single tree.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_model = RandomForestRegressor(n_estimators=300, max_depth=14, random_state=42, n_jobs=-1)
forest_model.fit(X_train, y_train)
forest_prediction = forest_model.predict(X_test)

forest_mae, forest_mape = score_model(y_test.values, forest_prediction)
results.append(("Random Forest", forest_mae, forest_mape))

print("Random Forest MAE:", round(forest_mae, 1), "MW")
print("Random Forest MAPE:", round(forest_mape, 2), "%")


## Gradient Boosting

This is similar to Random Forest, but instead of building all the trees independently, each new tree is trained specifically to fix the mistakes of the trees before it. It usually gets a bit more accuracy than Random Forest.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

boosting_model = GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42)
boosting_model.fit(X_train, y_train)
boosting_prediction = boosting_model.predict(X_test)

boosting_mae, boosting_mape = score_model(y_test.values, boosting_prediction)
results.append(("Gradient Boosting", boosting_mae, boosting_mape))

print("Gradient Boosting MAE:", round(boosting_mae, 1), "MW")
print("Gradient Boosting MAPE:", round(boosting_mape, 2), "%")


## LSTM — a neural network that reads data like a sentence

Every model so far looks at one row at a time. An **LSTM** (Long Short-Term Memory network) is different: it reads a whole *sequence* of hours in order,  like reading a sentence word by word and tries to learn patterns in how demand evolves over time, without us telling it what the lag features should be.

This takes a few extra setup steps:
1. Turn the data into "chunks" of 168 hours (1 week) each, since the LSTM needs a sequence to look at, not just a single row
2. Scale the numbers so they're all roughly the same size (neural networks train better this way)
3. Train it in rounds ("epochs"), checking after each round whether it's actually improving
4. Stop early if it stops improving, so it doesn't waste time or overfit

In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

lookback_hours = 168   # one week of history per sequence

lstm_feature_columns = [
    "demand_MW", "temperature_C", "humidity_pct", "dewpoint_C", "wind_speed_kmh",
    "solar_MW", "wind_MW", "renewable_MW",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos",
    "is_weekend", "is_holiday",
]


In [ ]:
# Set aside a small validation slice from the training data, to check progress while training
val_cutoff = cutoff_date - pd.Timedelta(days=45)

X_all = df[lstm_feature_columns].values.astype("float32")
y_all = df["target_t_plus_24h"].values.astype("float32")

valid_rows = np.arange(lookback_hours - 1, len(df))
row_times = df["timestamp"].values[valid_rows]

is_train_row = row_times < np.datetime64(val_cutoff)
is_val_row = (row_times >= np.datetime64(val_cutoff)) & (row_times < np.datetime64(cutoff_date))
is_test_row = row_times >= np.datetime64(cutoff_date)

print("Train sequences:", is_train_row.sum())
print("Validation sequences:", is_val_row.sum())
print("Test sequences:", is_test_row.sum())


In [ ]:
# Scale features and target so the neural network trains smoothly
train_row_numbers = valid_rows[is_train_row]
rows_needed_for_scaling = np.unique(np.concatenate(
    [np.arange(max(0, i - lookback_hours + 1), i + 1) for i in train_row_numbers]
))

feature_scaler = StandardScaler().fit(X_all[rows_needed_for_scaling])
X_scaled = feature_scaler.transform(X_all).astype("float32")

target_scaler = StandardScaler().fit(y_all[train_row_numbers].reshape(-1, 1))
y_scaled = target_scaler.transform(y_all.reshape(-1, 1)).astype("float32").ravel()

def make_sequences(row_numbers):
    sequences = np.stack([X_scaled[i - lookback_hours + 1 : i + 1] for i in row_numbers])
    targets = y_scaled[row_numbers]
    return torch.from_numpy(sequences), torch.from_numpy(targets)

X_train_seq, y_train_seq = make_sequences(valid_rows[is_train_row])
X_val_seq, y_val_seq = make_sequences(valid_rows[is_val_row])
X_test_seq, y_test_seq = make_sequences(valid_rows[is_test_row])
test_actual_mw = y_all[valid_rows[is_test_row]]


In [ ]:
class DemandLSTM(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.lstm = nn.LSTM(num_features, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2)
        self.output_layer = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x):
        lstm_output, _ = self.lstm(x)
        last_step = lstm_output[:, -1, :]   # only the final hour's "understanding" matters for our prediction
        return self.output_layer(last_step).squeeze(-1)

lstm_model = DemandLSTM(num_features=len(lstm_feature_columns))
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)
loss_function = nn.MSELoss()


In [ ]:
batch_size = 64
max_rounds = 40
patience = 5          # stop if we go this many rounds without improving
best_val_error = float("inf")
best_model_weights = None
rounds_without_improvement = 0

for round_number in range(1, max_rounds + 1):
    lstm_model.train()
    shuffled_order = torch.randperm(X_train_seq.shape[0])

    for start in range(0, X_train_seq.shape[0], batch_size):
        batch_index = shuffled_order[start : start + batch_size]
        batch_x = X_train_seq[batch_index]
        batch_y = y_train_seq[batch_index]

        optimizer.zero_grad()
        prediction = lstm_model(batch_x)
        loss = loss_function(prediction, batch_y)
        loss.backward()
        optimizer.step()

    # Check progress on the validation slice
    lstm_model.eval()
    with torch.no_grad():
        val_prediction_scaled = lstm_model(X_val_seq).numpy()
        val_prediction_mw = target_scaler.inverse_transform(val_prediction_scaled.reshape(-1, 1)).ravel()
        val_actual_mw = target_scaler.inverse_transform(y_val_seq.numpy().reshape(-1, 1)).ravel()
        val_error = np.mean(np.abs(val_prediction_mw - val_actual_mw))

    improved = val_error < best_val_error - 1
    print("Round", round_number, "- validation error:", round(val_error, 1), "MW", "(best so far!)" if improved else "")

    if improved:
        best_val_error = val_error
        best_model_weights = {k: v.clone() for k, v in lstm_model.state_dict().items()}
        rounds_without_improvement = 0
    else:
        rounds_without_improvement += 1

    if rounds_without_improvement >= patience:
        print("Stopping early - no improvement for", patience, "rounds")
        break


In [ ]:
# Load the best version of the model (not necessarily the very last one) and test it
lstm_model.load_state_dict(best_model_weights)
lstm_model.eval()
with torch.no_grad():
    test_prediction_scaled = lstm_model(X_test_seq).numpy()
lstm_prediction = target_scaler.inverse_transform(test_prediction_scaled.reshape(-1, 1)).ravel()

lstm_mae, lstm_mape = score_model(test_actual_mw, lstm_prediction)
results.append(("LSTM", lstm_mae, lstm_mape))

print("LSTM MAE:", round(lstm_mae, 1), "MW")
print("LSTM MAPE:", round(lstm_mape, 2), "%")


## Compare every model side by side

In [ ]:
results_table = pd.DataFrame(results, columns=["Model", "MAE (MW)", "MAPE (%)"])
results_table["Improvement vs baseline"] = (
    (results_table["MAE (MW)"][0] - results_table["MAE (MW)"]) / results_table["MAE (MW)"][0] * 100
).round(1)

results_table


In [ ]:
plt.figure(figsize=(8, 4))
colors = ["gray" if name == "Baseline (persistence)" else "steelblue" for name in results_table["Model"]]
plt.bar(results_table["Model"], results_table["MAE (MW)"], color=colors)
plt.title("Model Comparison - Lower is Better")
plt.ylabel("MAE (MW)")
plt.xticks(rotation=20)
plt.show()


## Look at the winning model's predictions

We'll plot the real demand against Gradient Boosting's predictions for the whole test period.

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(test["timestamp"], y_test, label="Actual demand", linewidth=1.5)
plt.plot(test["timestamp"], boosting_prediction, label="Predicted (Gradient Boosting)", linewidth=1.2, alpha=0.8)
plt.title("Predicted vs. Actual Demand")
plt.xlabel("Date")
plt.ylabel("Demand (MW)")
plt.legend()
plt.show()


## Important Features

Gradient Boosting can tell us which features it relied on most. This is a great way to check that the model learned something sensible, not something random.

In [ ]:
importance = pd.Series(boosting_model.feature_importances_, index=feature_columns)
top_10_features = importance.sort_values(ascending=False).head(10)

plt.figure(figsize=(7, 5))
top_10_features.sort_values().plot(kind="barh", color="teal")
plt.title("Top 10 Most Important Features")
plt.xlabel("Importance")
plt.show()


## Conclusion

- We tried 4 real models plus a simple baseline, and picked the winner by comparing actual error numbers — not by guessing which sounded fanciest.
- **Gradient Boosting** usually comes out on top for this kind of problem.
- Notice that the LSTM, despite being the most complex model here, doesn't automatically win — neural networks need a *lot* of data to show their strength, and a year of hourly data is relatively small for one.
- Next: open **EnergyPredict_Renewable_and_Dispatch.ipynb** to forecast solar+wind output and turn both forecasts into a grid-stress recommendation.